In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
from torchvision import transforms

In [2]:
class ImagesDataset(Dataset):
    def __init__(self, image_folder, transform=None):
        self.image_folder = image_folder
        self.image_paths = [
            os.path.join(image_folder, fname)
            for fname in os.listdir(image_folder)
            if fname.endswith(".JPEG")
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = self.image_paths[index]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, img_path

In [3]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)

In [4]:
image_folder = "data/test_all"

dataset = ImagesDataset(image_folder, transform)

dataloader = DataLoader(dataset, batch_size=128, shuffle=False)

In [5]:
from model import SEBlock, MBConvBlock, ConvNet, LitConvNet

import lightning as L

ckpt_path = "images-classification/j6aujzlv/checkpoints/best-model-epoch=09.ckpt"

model = LitConvNet.load_from_checkpoint(ckpt_path)

In [6]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"


model.eval()

predictions = []
image_paths = []

with torch.inference_mode():
    for images, paths in dataloader:
        images = images.to(device)

        logits = model(images)
        preds = logits.argmax(dim=1)

        predictions.extend(preds.cpu().numpy())
        image_paths.extend(paths)

In [7]:
assert len(dataset) == len(predictions)

In [8]:
import pandas as pd

# os.path.basename(image_paths[0])
results = pd.DataFrame({"Image Path": image_paths, "Class": predictions})
results

,Image Path,Class
0,data/test_all/8885116932060749.JPEG,37
1,data/test_all/9040528456606877.JPEG,21
2,data/test_all/6501173720479193.JPEG,1
3,data/test_all/6466322132242255.JPEG,25
4,data/test_all/8926758762754503.JPEG,40
...,...,...
9995,data/test_all/05423695111982274.JPEG,14
9996,data/test_all/34890558640617564.JPEG,9
9997,data/test_all/11083798003819756.JPEG,27
9998,data/test_all/5568873749583029.JPEG,25


In [9]:
results["Image Path"] = results["Image Path"].apply(os.path.basename)
results

,Image Path,Class
0,8885116932060749.JPEG,37
1,9040528456606877.JPEG,21
2,6501173720479193.JPEG,1
3,6466322132242255.JPEG,25
4,8926758762754503.JPEG,40
...,...,...
9995,05423695111982274.JPEG,14
9996,34890558640617564.JPEG,9
9997,11083798003819756.JPEG,27
9998,5568873749583029.JPEG,25


In [11]:
results.to_csv("pred.csv", index=False, header=False)